<a href="https://colab.research.google.com/github/mohiuddin2806-lang/machine-learning-/blob/main/CN7031_week_3_u2975262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ▶ 1.1 Set up Hadoop 3.5.0 (one NameNode + one DataNode) and Apache Sqoop 1.4.7
# Import paths, operating-system utilities, download and checksum tools.
from pathlib import Path
import os
import subprocess
import urllib.request
import urllib.error
import hashlib
import re
import tarfile
import time

# Import XML tools to create Hadoop configuration files.
from xml.etree import ElementTree as ET

# Import the HTTP client used to check the NameNode and DataNode web services.
import requests

# Set a persistent path within the current temporary Colab runtime.
LAB = Path('/content/cn7031-week3-sqoop')
# Create the working directory when it does not exist.
LAB.mkdir(parents=True, exist_ok=True)
# Specify the Hadoop, Sqoop and Java paths.
HADOOP = LAB / 'hadoop-3.5.0'
SQOOP = LAB / 'sqoop-1.4.7.bin__hadoop-2.6.0'
JAVA = Path('/usr/lib/jvm/java-17-openjdk-amd64')
# Maven Central hosts the extra Java libraries Sqoop needs.
MAVEN = 'https://repo1.maven.org/maven2'

# Run a system command and stop with a useful error if it fails.
def checked(args, timeout=600):
    # Capture standard output and errors as readable text.
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout)
    # Check the exit status before using the command's result.
    if p.returncode:
        raise RuntimeError(p.stdout[-5000:])
    # Return the printed output to the caller.
    return p.stdout

# Match a digest's length (in hexadecimal characters) to the algorithm that produced it.
DIGEST_BY_LENGTH = {32: 'md5', 40: 'sha1', 64: 'sha256', 128: 'sha512'}

# Read the first checksum file a publisher provides next to a download.
def published_digest(url, suffixes):
    # Remember the filename so it can be removed from the checksum text.
    name = url.rsplit('/', 1)[-1]
    for suffix in suffixes:
        try:
            # Download the small checksum file, e.g. file.tar.gz.sha512.
            with urllib.request.urlopen(url + suffix, timeout=60) as response:
                text = response.read().decode(errors='replace')
        except (urllib.error.URLError, TimeoutError):
            # Try the next checksum format if this one is not published.
            continue
        # Remove the filename, then keep only the hexadecimal groups.
        tokens = re.split(r'[\s:=()*]+', text.replace(name, ' '))
        digest = ''.join(t for t in tokens if re.fullmatch(r'[0-9a-fA-F]+', t)).lower()
        # Return the algorithm and digest when the length is recognised.
        if len(digest) in DIGEST_BY_LENGTH:
            return DIGEST_BY_LENGTH[len(digest)], digest
    # No usable checksum file was found.
    return None, None

# Download a file and keep it only if it matches the published checksum.
def download_verified(url, dest, suffixes, required=True):
    # Download into a temporary name so a failed check leaves nothing behind.
    partial = dest.with_name(dest.name + '.part')
    urllib.request.urlretrieve(url, partial)
    # Look up the publisher's checksum for this exact file.
    algorithm, expected = published_digest(url, suffixes)
    if algorithm is None:
        # Stop, unless this download is allowed to proceed without a checksum.
        if required:
            partial.unlink()
            raise RuntimeError(f'No checksum published for {url}. Stop here.')
        print(f'  Note: no checksum file found for {dest.name}; continuing without one.')
    else:
        # Calculate the digest of the downloaded file with the same algorithm.
        with partial.open('rb') as stream:
            actual = hashlib.file_digest(stream, algorithm).hexdigest()
        # Abort setup if the digests do not match.
        if actual != expected:
            partial.unlink()
            raise RuntimeError(f'Download checksum failed for {dest.name}. Stop here.')
        print(f'  {algorithm.upper()} checksum verified: {dest.name}')
    # Give the verified file its final name.
    partial.replace(dest)

# Install Java 17 only when it is not already available.
# Sqoop compiles Java code at run time, so it needs the compiler (javac) as well as java.
if not (JAVA / 'bin/javac').exists():
    # Update package metadata for the Colab VM.
    print('Installing Java …')
    checked(['apt-get', 'update', '-qq'])
    # Install the headless JDK required by Hadoop and Sqoop.
    checked(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'])

# Download and verify Hadoop only once per current VM.
if not (HADOOP / '.verified').exists():
    # Identify the archive destination and Apache download URL.
    print('Downloading Hadoop from Apache (about 554 MB) …')
    archive = LAB / 'hadoop-3.5.0.tar.gz'
    base = 'https://downloads.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz'
    # Keep track of progress so the display is not too noisy.
    last = [-1]

    # Report download progress in roughly 20% increments.
    def progress(count, block, total):
        # A known total size lets us calculate the percentage.
        if total > 0:
            percent = min(100, int(count * block * 100 / total))
            # Print only when a new progress interval is reached.
            if percent // 20 > last[0]:
                print(f'  Download: {percent}%')
                last[0] = percent // 20

    # Try the official mirror; use the Apache archive if unavailable.
    try:
        urllib.request.urlretrieve(base, archive, reporthook=progress)
    except (urllib.error.URLError, TimeoutError):
        # Use an archive URL instead of a missing current-release URL.
        base = base.replace('downloads.apache.org', 'archive.apache.org/dist')
        urllib.request.urlretrieve(base, archive, reporthook=progress)

    # Read Apache's published SHA-512 digest for this exact archive.
    with urllib.request.urlopen(base + '.sha512', timeout=60) as response:
        digest = re.findall(r'(?i)\b[0-9a-f]{128}\b', response.read().decode())
    # Calculate the SHA-512 digest of the downloaded file.
    with archive.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha512').hexdigest()
    # Abort setup if the digests do not match.
    assert len(digest) == 1 and digest[0].lower() == actual, 'Download checksum failed. Stop here.'
    # Extract the verified archive safely.
    print('Checksum verified. Installing Hadoop …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    # Record successful verification and remove the compressed archive.
    (HADOOP / '.verified').write_text(actual)
    archive.unlink()

# Download and verify Sqoop only once per current VM.
if not (SQOOP / '.verified').exists():
    # Sqoop is a retired Apache project, so it is only kept in the Apache archive.
    print('Downloading Apache Sqoop 1.4.7 …')
    archive = LAB / 'sqoop-1.4.7.tar.gz'
    download_verified('https://archive.apache.org/dist/sqoop/1.4.7/sqoop-1.4.7.bin__hadoop-2.6.0.tar.gz',
                      archive, ('.sha512', '.sha1', '.md5'), required=False)
    # Extract the archive safely.
    print('Installing Sqoop …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    # Make sure Sqoop's launcher scripts are executable.
    for script in (SQOOP / 'bin').iterdir():
        script.chmod(script.stat().st_mode | 0o111)
    # Record successful installation and remove the compressed archive.
    (SQOOP / '.verified').write_text('ok')
    archive.unlink()

# Add the MySQL JDBC driver, commons-lang 2 and commons-cli 1.4 to Sqoop's library folder.
EXTRA_JARS = {
    'mysql-connector-j-8.0.33.jar': f'{MAVEN}/com/mysql/mysql-connector-j/8.0.33/mysql-connector-j-8.0.33.jar',
    'commons-lang-2.6.jar': f'{MAVEN}/commons-lang/commons-lang/2.6/commons-lang-2.6.jar',
    'commons-cli-1.4.jar': f'{MAVEN}/commons-cli/commons-cli/1.4/commons-cli-1.4.jar',
}
for name, url in EXTRA_JARS.items():
    # Download each library once; Maven Central publishes a .sha1 checksum for every file.
    if not (SQOOP / 'lib' / name).exists():
        print(f'Downloading {name} …')
        download_verified(url, SQOOP / 'lib' / name, ('.sha1',))

# Sqoop reads conf/sqoop-env.sh at every start: load commons-cli 1.4 ahead of Hadoop's newer copy,
# because Sqoop 1.4.7 reads its options with a commons-cli method that later versions removed.
(SQOOP / 'conf' / 'sqoop-env.sh').write_text(
    f'export HADOOP_CLIENT_OPTS="$HADOOP_CLIENT_OPTS -Xbootclasspath/a:{SQOOP / "lib" / "commons-cli-1.4.jar"}"\n')

# Create an empty folder to stand in for HBase, HCatalog, Accumulo and ZooKeeper (not used in this lab).
NOT_INSTALLED = LAB / 'not-installed'
NOT_INSTALLED.mkdir(exist_ok=True)

# Set Java, Hadoop and Sqoop environment variables for child processes (!, %%bash, sqoop).
os.environ.update(JAVA_HOME=str(JAVA), HADOOP_HOME=str(HADOOP),
                  HADOOP_CONF_DIR=str(HADOOP / 'etc/hadoop'),
                  HADOOP_LOG_DIR=str(LAB / 'logs'), HADOOP_PID_DIR=str(LAB / 'pids'),
                  # Sqoop finds Hadoop's client libraries through these two variables.
                  HADOOP_COMMON_HOME=str(HADOOP), HADOOP_MAPRED_HOME=str(HADOOP),
                  SQOOP_HOME=str(SQOOP),
                  # Pointing these at an empty folder silences Sqoop's "not installed" warnings.
                  HBASE_HOME=str(NOT_INSTALLED), HCAT_HOME=str(NOT_INSTALLED),
                  ACCUMULO_HOME=str(NOT_INSTALLED), ZOOKEEPER_HOME=str(NOT_INSTALLED))
# Give client programs (hdfs dfs, sqoop) 1 GB of memory and access to JDK internals that older Hadoop code uses.
os.environ['HADOOP_CLIENT_OPTS'] = ('-Xmx1g --add-opens=java.base/java.lang=ALL-UNNAMED '
                                    '--add-opens=java.base/java.util=ALL-UNNAMED '
                                    '--add-opens=java.base/java.nio=ALL-UNNAMED '
                                    '--add-opens=java.base/sun.nio.ch=ALL-UNNAMED')
# Folder for the Java code Sqoop generates and compiles for each table it copies.
CODEGEN = LAB / 'codegen'
CODEGEN.mkdir(exist_ok=True)
os.environ['CODEGEN_DIR'] = str(CODEGEN)

# Lab wrapper for the sqoop command. Tools that generate Java code (import, import-all-tables, export)
# write it to CODEGEN_DIR, and CODEGEN_DIR is put on Hadoop's classpath. In local mode the map tasks
# run inside the Sqoop process and can only load each table's class from that classpath.
WRAPPER_DIR = LAB / 'bin'
WRAPPER_DIR.mkdir(exist_ok=True)
(WRAPPER_DIR / 'sqoop').write_text(f"""#!/bin/bash
# CN7031 lab wrapper for Sqoop (see Step 1.1).
export CODEGEN_DIR="${{CODEGEN_DIR:-{CODEGEN}}}"
mkdir -p "$CODEGEN_DIR"
export HADOOP_CLASSPATH="$CODEGEN_DIR${{HADOOP_CLASSPATH:+:$HADOOP_CLASSPATH}}"
case "$1" in
  import|import-all-tables|export)
    tool="$1"; shift
    # For MySQL, read column names in table order (see the column-order fix below).
    extra=()
    case " $* " in
      *" --connection-manager"*|*" --driver"*) ;;
      *jdbc:mysql:*) extra=(--connection-manager cn7031.OrderedMySQLManager) ;;
    esac
    exec "{SQOOP}/bin/sqoop" "$tool" --outdir "$CODEGEN_DIR" --bindir "$CODEGEN_DIR" "${{extra[@]}}" "$@" ;;
  *)
    exec "{SQOOP}/bin/sqoop" "$@" ;;
esac
""")
(WRAPPER_DIR / 'sqoop').chmod(0o755)

# Show Sqoop's own progress messages, but only warnings and errors from Hadoop's internals,
# so each Sqoop command prints a readable summary instead of hundreds of MapReduce log lines.
log4j = HADOOP / 'etc/hadoop/log4j.properties'
QUIET = ['log4j.logger.org.apache.hadoop=WARN',
         'log4j.logger.org.apache.sqoop=INFO',
         # Harmless in local mode: Hadoop tries to link each job's libraries into the current folder
         # and warns when it cannot. The job still succeeds, so hide these "failed" warnings.
         'log4j.logger.org.apache.hadoop.fs.FileUtil=ERROR',
         'log4j.logger.org.apache.hadoop.mapred.LocalDistributedCacheManager=ERROR',
         'log4j.logger.org.apache.hadoop.metrics2.impl.MetricsSystemImpl=ERROR']
if log4j.exists():
    text = log4j.read_text()
    missing = [line for line in QUIET if line not in text]
    if missing:
        log4j.write_text(text + '\n# CN7031: quieter client output\n' + '\n'.join(missing) + '\n')

# Put the lab wrapper, Sqoop, Hadoop and Java executables on the shell PATH (wrapper first).
os.environ['PATH'] = f'{WRAPPER_DIR}:{SQOOP}/bin:{HADOOP}/bin:{JAVA}/bin:' + os.environ['PATH']

# Column-order fix. Sqoop 1.4.7 reads a MySQL table's column names from INFORMATION_SCHEMA without
# ORDER BY. MySQL 8 does not guarantee an order without one and sometimes returns them alphabetically,
# so Sqoop would write the columns in the wrong order (e.g. 2|1|Football instead of 1|2|Football).
# This small connection manager adds ORDER BY ORDINAL_POSITION; the sqoop wrapper above uses it.
FIX_JAR = SQOOP / 'lib' / 'cn7031-ordered-mysql-manager.jar'
if not FIX_JAR.exists():
    fix_src = LAB / 'sqoop-fix' / 'src' / 'cn7031'
    fix_cls = LAB / 'sqoop-fix' / 'classes'
    fix_src.mkdir(parents=True, exist_ok=True)
    fix_cls.mkdir(parents=True, exist_ok=True)
    (fix_src / 'OrderedMySQLManager.java').write_text("""package cn7031;

import com.cloudera.sqoop.SqoopOptions;
import org.apache.sqoop.manager.MySQLManager;

/** MySQLManager that lists a table's columns in table order (Sqoop 1.4.7 omits ORDER BY). */
public class OrderedMySQLManager extends MySQLManager {
  public OrderedMySQLManager(final SqoopOptions opts) {
    super(opts);
  }

  @Override
  protected String getListColumnsQuery(String tableName) {
    return super.getListColumnsQuery(tableName) + " ORDER BY ORDINAL_POSITION";
  }
}
""")
    # Compile against Sqoop's own jar, its libraries and Hadoop's client libraries, then package it.
    hadoop_cp = subprocess.run([str(HADOOP / 'bin/hadoop'), 'classpath'], capture_output=True,
                               text=True, check=True).stdout.strip().splitlines()[-1]
    sqoop_jar = next(SQOOP.glob('sqoop-1.4.7*.jar'))
    checked([str(JAVA / 'bin/javac'), '-nowarn', '-cp', f'{sqoop_jar}:{SQOOP}/lib/*:{hadoop_cp}',
             '-d', str(fix_cls), str(fix_src / 'OrderedMySQLManager.java')])
    checked([str(JAVA / 'bin/jar'), 'cf', str(FIX_JAR), '-C', str(fix_cls), '.'])
# Make separate folders for namespace metadata, blocks, logs and process IDs.
for name in ('namenode', 'datanode', 'logs', 'pids'):
    (LAB / name).mkdir(exist_ok=True)

# Write a Hadoop XML configuration file from a property dictionary.
def config(name, values):
    # Create the configuration root element.
    root = ET.Element('configuration')
    # Add each property name and its value.
    for key, value in values.items():
        property_node = ET.SubElement(root, 'property')
        ET.SubElement(property_node, 'name').text = key
        ET.SubElement(property_node, 'value').text = str(value)
    # Save the resulting configuration into Hadoop's config directory.
    ET.ElementTree(root).write(HADOOP / 'etc/hadoop' / name, encoding='utf-8')

# Point HDFS clients at the local NameNode using port 9000.
config('core-site.xml', {
    'fs.defaultFS': 'hdfs://127.0.0.1:9000',
    'hadoop.tmp.dir': str(LAB / 'tmp')
})
# Assign local storage directories and a default replication factor of one.
config('hdfs-site.xml', {
    'dfs.namenode.name.dir': (LAB / 'namenode').as_uri(),
    'dfs.datanode.data.dir': (LAB / 'datanode').as_uri(),
    'dfs.replication': 1,
    # Bind RPC and health-report endpoints to the VM's loopback address.
    'dfs.namenode.rpc-address': '127.0.0.1:9000',
    'dfs.namenode.rpc-bind-host': '127.0.0.1',
    'dfs.namenode.http-address': '127.0.0.1:9870',
    'dfs.namenode.http-bind-host': '127.0.0.1',
    # Keep DataNode transfer, HTTP and IPC endpoints on localhost.
    'dfs.datanode.address': '127.0.0.1:9866',
    'dfs.datanode.http.address': '127.0.0.1:9864',
    'dfs.datanode.ipc.address': '127.0.0.1:9867',
    'dfs.datanode.hostname': '127.0.0.1',
    'dfs.client.use.datanode.hostname': 'true',
    'dfs.datanode.use.datanode.hostname': 'true',
})
# Run Sqoop's MapReduce jobs in local mode (inside the Sqoop process, no YARN cluster).
config('mapred-site.xml', {
    'mapreduce.framework.name': 'local',
    # Keep each job's temporary staging files in one HDFS folder.
    'mapreduce.jobtracker.staging.root.dir': '/tmp/hadoop-staging',
})

# Read Hadoop's environment shell script before adding Java and heap settings.
env = HADOOP / 'etc/hadoop/hadoop-env.sh'
# Avoid appending duplicate configuration lines on reruns.
if 'export HADOOP_HEAPSIZE_MAX=512' not in env.read_text():
    env.write_text(env.read_text() +
                   f'\n# CN7031 HDFS environment\nexport JAVA_HOME={JAVA}\nexport HADOOP_HEAPSIZE_MAX=512\n')

# Format only when no NameNode filesystem already exists.
if not (LAB / 'namenode/current/VERSION').exists():
    # Never silently overwrite an unexpectedly populated storage directory.
    assert all(not any((LAB / name).iterdir()) for name in ('namenode', 'datanode')), \
        'Existing storage found. Do not reformat it.'
    # Give the new HDFS namespace a fixed educational cluster ID.
    checked(['hdfs', 'namenode', '-format', '-nonInteractive', '-clusterid', 'CID-cn7031-week3'])

# Start a service only if its local HTTP endpoint is not responding.
for daemon, port in [('namenode', 9870), ('datanode', 9864)]:
    try:
        # Raise an error on a missing or unhealthy service endpoint.
        requests.get(f'http://127.0.0.1:{port}/jmx', timeout=2).raise_for_status()
    except requests.RequestException:
        # Start the selected Hadoop daemon in the background.
        checked(['hdfs', '--daemon', 'start', daemon])

# Wait until the NameNode reports exactly one live DataNode.
for attempt in range(45):
    try:
        # Query NameNode's JMX metrics over its local HTTP endpoint.
        response = requests.get('http://127.0.0.1:9870/jmx',
                                params={'qry': 'Hadoop:service=NameNode,name=FSNamesystemState'},
                                timeout=3)
        response.raise_for_status()
        # Exit the retry loop when the teaching cluster is ready.
        if response.json()['beans'][0]['NumLiveDataNodes'] == 1:
            break
    except (requests.RequestException, KeyError, IndexError):
        # Retry while the services finish their startup sequence.
        pass
    # Pause two seconds before asking again.
    time.sleep(2)
else:
    # Stop if the DataNode never becomes available.
    raise RuntimeError('Startup failed. Inspect the Hadoop logs in /content/cn7031-week3-sqoop/logs.')

# Wait until the NameNode has left safe mode, allowing writes.
checked(['hdfs', 'dfsadmin', '-safemode', 'wait'], timeout=90)
# Ask Sqoop for its version; this proves Sqoop can load Hadoop and its libraries.
version = next((line for line in checked(['sqoop', 'version'], timeout=180).splitlines()
                if line.startswith('Sqoop ')), 'Sqoop (version not found)')
# Ask for help on one tool; this makes Sqoop parse its options, which needs commons-cli 1.4.
checked(['sqoop', 'help', 'list-databases'], timeout=180)
# Confirm that HDFS and Sqoop are usable before the exercises begin.
print('\nHDFS is ready ✓')
print('Hadoop 3.5.0 · NameNode running · 1 DataNode running')
print(f'{version} is ready ✓ · MySQL JDBC driver 8.0.33 · column-order fix installed')

Installing Java …
  Download: 0%
  Download: 20%
  Download: 40%
  Download: 60%
  Download: 80%
  Download: 100%
Checksum verified. Installing Hadoop …
  MD5 checksum verified: sqoop-1.4.7.tar.gz
Installing Sqoop …
  SHA1 checksum verified: mysql-connector-j-8.0.33.jar
  SHA1 checksum verified: commons-lang-2.6.jar
  SHA1 checksum verified: commons-cli-1.4.jar

HDFS is ready ✓
Hadoop 3.5.0 · NameNode running · 1 DataNode running
Sqoop 1.4.7 is ready ✓ · MySQL JDBC driver 8.0.33 · column-order fix installed


In [2]:
# @title ▶ 1.2 Install MySQL 8, set the root password and load retail_db
# Import process, checksum, download and path tools.
import os
import time
import hashlib
import subprocess
import urllib.request
from pathlib import Path

# Run a command quietly and return its result (check=True stops on errors).
def run(args, check=True, **kwargs):
    return subprocess.run(args, text=True, capture_output=True, check=check, **kwargs)

# Install the MySQL server package only when it is not already available.
if run(['which', 'mysqld'], check=False).returncode:
    # Update package metadata, then install without interactive questions.
    print('Installing MySQL server (about 1 minute) …')
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'mysql-server'],
        env={**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'})

# Start the server (harmless if it is already running).
run(['service', 'mysql', 'start'], check=False)
# Wait until the server answers.
for attempt in range(30):
    if run(['mysqladmin', '--no-defaults', 'ping'], check=False).returncode == 0:
        break
    time.sleep(1)
else:
    # Stop if MySQL never starts.
    raise RuntimeError('MySQL did not start. Try Runtime → Restart session, then rerun Section 1.')

# A fresh install lets Linux root log in without a password; set the tutorial password once.
if run(['mysql', '--no-defaults', '-u', 'root', '-e', 'SELECT 1'], check=False).returncode == 0:
    run(['mysql', '--no-defaults', '-u', 'root', '-e',
         "ALTER USER 'root'@'localhost' IDENTIFIED WITH caching_sha2_password BY 'cloudera';"])

# Store the login in an option file so later mysql commands need no password prompt.
my_cnf = Path.home() / '.my.cnf'
my_cnf.write_text('[client]\nuser=root\npassword=cloudera\ndefault-character-set=utf8mb4\n')
# Make the option file readable by its owner only.
my_cnf.chmod(0o600)

# Load retail_db unless it is already complete (68,883 orders).
orders = run(['mysql', '-N', '-e', 'SELECT COUNT(*) FROM retail_db.orders'], check=False)
if orders.returncode or orders.stdout.strip() != '68883':
    # Use an uploaded copy if present; otherwise download the dump.
    dump = Path('/content/retail_db.sql')
    if not dump.exists():
        print('Downloading the retail_db SQL dump (about 10 MB) …')
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/dgadiraju/retail_db/master/create_db.sql', dump)
    # Check the dump is byte-for-byte the copy this notebook was tested with.
    with dump.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    assert actual == '90e630cec0d0030de82ccd2c49ad61668e7b9965f0a39f2639b233a56dede5ea', \
        'retail_db.sql is not the expected file. Delete /content/retail_db.sql and rerun this cell.'
    # Create an empty database and run the dump's SQL statements into it.
    print('Loading retail_db into MySQL …')
    run(['mysql', '-e', 'DROP DATABASE IF EXISTS retail_db; CREATE DATABASE retail_db;'])
    with dump.open() as sql:
        subprocess.run(['mysql', 'retail_db'], stdin=sql, check=True)

# Confirm that MySQL is usable, and report each table's row count.
version = run(['mysql', '-N', '-e', 'SELECT VERSION()']).stdout.strip()
print(f'\nMySQL is ready ✓ (server {version}, user root, password cloudera)')
for table in ['categories', 'customers', 'departments', 'order_items', 'orders', 'products']:
    rows = run(['mysql', '-N', '-e', f'SELECT COUNT(*) FROM retail_db.{table}']).stdout.strip()
    print(f'  retail_db.{table:<12} {int(rows):>8,} rows')

# Notebook shortcut: a %%mysql cell sends its lines to the mysql client, like typing them at the mysql> prompt.
from IPython.core.magic import register_cell_magic

@register_cell_magic
def mysql(line, cell):
    # -t draws result tables; -vv echoes each statement and reports "N rows in set" or "Query OK".
    result = subprocess.run(['mysql', '-t', '-vv', *line.split()], input=cell, text=True, capture_output=True)
    print(result.stdout + result.stderr, end='')

print('\n%%mysql cells are ready ✓')

Installing MySQL server (about 1 minute) …
Loading retail_db into MySQL …

MySQL is ready ✓ (server 8.0.46-0ubuntu0.24.04.4, user root, password cloudera)
  retail_db.categories         58 rows
  retail_db.customers      12,435 rows
  retail_db.departments         6 rows
  retail_db.order_items   172,198 rows
  retail_db.orders         68,883 rows
  retail_db.products        1,345 rows

%%mysql cells are ready ✓


In [3]:
# List local Java processes to confirm that both HDFS services started.
!jps

28995 Jps
24103 DataNode
24042 NameNode


In [4]:
# Ask the MySQL server whether it is alive and how long it has been running.
!mysqladmin status

Uptime: 543  Threads: 2  Questions: 132  Slow queries: 0  Opens: 167  Flush tables: 3  Open tables: 80  Queries per second avg: 0.243


In [6]:
%%mysql
SHOW DATABASES;

--------------
SHOW DATABASES
--------------

+--------------------+
| Database           |
+--------------------+
| information_schema |
| mysql              |
| performance_schema |
| retail_db          |
| sys                |
+--------------------+
5 rows in set

Bye


In [7]:
%%mysql
USE retail_db;
SHOW TABLES;

--------------
SHOW TABLES
--------------

+---------------------+
| Tables_in_retail_db |
+---------------------+
| categories          |
| customers           |
| departments         |
| order_items         |
| orders              |
| products            |
+---------------------+
6 rows in set

Bye


In [8]:
%%mysql
USE retail_db;
DESCRIBE customers;

--------------
DESCRIBE customers
--------------

+-------------------+--------------+------+-----+---------+----------------+
| Field             | Type         | Null | Key | Default | Extra          |
+-------------------+--------------+------+-----+---------+----------------+
| customer_id       | int          | NO   | PRI | NULL    | auto_increment |
| customer_fname    | varchar(45)  | NO   |     | NULL    |                |
| customer_lname    | varchar(45)  | NO   |     | NULL    |                |
| customer_email    | varchar(45)  | NO   |     | NULL    |                |
| customer_password | varchar(45)  | NO   |     | NULL    |                |
| customer_street   | varchar(255) | NO   |     | NULL    |                |
| customer_city     | varchar(45)  | NO   |     | NULL    |                |
| customer_state    | varchar(45)  | NO   |     | NULL    |                |
| customer_zipcode  | varchar(45)  | NO   |     | NULL    |                |
+-------------------+-----

In [9]:
%%mysql
USE retail_db;
SELECT 'categories' AS table_name, COUNT(*) AS row_count FROM categories
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'departments', COUNT(*) FROM departments
UNION ALL
SELECT 'order's, COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'products', COUNT(*) FROM products;

--------------
SELECT 'categories' AS table_name, COUNT(*) AS row_count FROM categories
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'departments', COUNT(*) FROM departments
UNION ALL
SELECT 'order's, COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'products', COUNT(*) FROM products
--------------

+-------------+-----------+
| table_name  | row_count |
+-------------+-----------+
| categories  |        58 |
| customers   |     12435 |
| departments |         6 |
| order       |     68883 |
| order_items |    172198 |
| products    |      1345 |
+-------------+-----------+
6 rows in set

Bye


In [10]:
# @title
%env MYSQL_JDBC=jdbc:mysql://localhost:3306/?useSSL=false&allowPublicKeyRetrieval=true&zeroDateTimeBehavior=CONVERT_TO_NULL
%env RETAIL_JDBC=jdbc:mysql://localhost:3306/retail_db?useSSL=false&allowPublicKeyRetrieval=true&zeroDateTimeBehavior=CONVERT_TO_NULL

env: MYSQL_JDBC=jdbc:mysql://localhost:3306/?useSSL=false&allowPublicKeyRetrieval=true&zeroDateTimeBehavior=CONVERT_TO_NULL
env: RETAIL_JDBC=jdbc:mysql://localhost:3306/retail_db?useSSL=false&allowPublicKeyRetrieval=true&zeroDateTimeBehavior=CONVERT_TO_NULL


In [12]:
# List the databases the root account can see, this time through Sqoop and JDBC.
# --connect gives the connection string ($MYSQL_JDBC points at the server, no database).
# --username and --password give the login.
%%bash
sqoop list-tables \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera

categories
customers
departments
order_items
orders
products


2026-10-20 16:16:12,814 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 16:16:12,895 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 16:16:13,273 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.


In [14]:
%%bash
sqoop eval \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--query "SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY order_status ORDER BY orders DESC"

----------------------------------------------
| order_status         | orders              | 
----------------------------------------------
| COMPLETE             | 22899               | 
| PENDING_PAYMENT      | 15030               | 
| PROCESSING           | 8275                | 
| PENDING              | 7610                | 
| CLOSED               | 7556                | 
| ON_HOLD              | 3798                | 
| SUSPECTED_FRAUD      | 1558                | 
| CANCELED             | 1428                | 
| PAYMENT_REVIEW       | 729                 | 
----------------------------------------------


2026-10-20 16:18:52,201 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 16:18:52,300 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 16:18:52,623 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.


In [21]:
!hdfs dfs -rm -r -f /user/hive/warehouse

Deleted /user/hive/warehouse


In [51]:
%%bash
# Ensure Hadoop services are started before running Sqoop
hdfs --daemon start namenode 2>/dev/null || true
hdfs --daemon start datanode 2>/dev/null || true
hdfs dfsadmin -safemode wait 2>/dev/null || true

# Remove existing warehouse files to avoid conflicts
hdfs dfs -rm -r -f /user/hive/warehouse

sqoop import-all-tables \
-m 2 \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--fields-terminated-by '\t' \
--warehouse-dir /user/hive/warehouse

Safe mode is OFF
Deleted /user/hive/warehouse


2026-10-20 17:26:43,424 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 17:26:43,504 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 17:26:43,676 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
2026-10-20 17:26:45,066 INFO tool.CodeGenTool: Beginning code generation
2026-10-20 17:26:45,274 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `categories` AS t LIMIT 1
2026-10-20 17:26:45,401 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `categories` AS t LIMIT 1
2026-10-20 17:26:45,435 INFO orm.CompilationManager: HADOOP_MAPRED_HOME is /content/cn7031-week3-sqoop/hadoop-3.5.0
Note: /content/cn7031-week3-sqoop/codegen/categories.

In [23]:
!hdfs dfs -ls /user/hive/warehouse/

Found 6 items
drwxr-xr-x   - root supergroup          0 2026-10-20 16:28 /user/hive/warehouse/categories
drwxr-xr-x   - root supergroup          0 2026-10-20 16:28 /user/hive/warehouse/customers
drwxr-xr-x   - root supergroup          0 2026-10-20 16:28 /user/hive/warehouse/departments
drwxr-xr-x   - root supergroup          0 2026-10-20 16:29 /user/hive/warehouse/order_items
drwxr-xr-x   - root supergroup          0 2026-10-20 16:29 /user/hive/warehouse/orders
drwxr-xr-x   - root supergroup          0 2026-10-20 16:29 /user/hive/warehouse/products


In [24]:
!hdfs dfs -ls /user/hive/warehouse/categories/

Found 2 items
-rw-r--r--   1 root supergroup          0 2026-10-20 16:28 /user/hive/warehouse/categories/_SUCCESS
-rw-r--r--   1 root supergroup       1029 2026-10-20 16:28 /user/hive/warehouse/categories/part-m-00000


In [28]:
!hdfs dfs -cat /user/hive/warehouse/categories/part-m-00000 | head -n  5

1l2lFootball
2l2lSoccer
3l2lBaseball & Softball
4l2lBasketball
5l2lLacrosse


In [29]:
!hdfs dfs -cat /user/hive/warehouse/products/part-m-* | grep '^685'

685l31lTaylorMade SLDR Irons - (Steel) 4-PW, AWll899.99lhttp://images.acmesports.sports/TaylorMade+SLDR+Irons+-+%28Steel%29+4-PW%2C+AW


In [30]:
%%bash
for t in categories customers departments order_items order products; do
hdfs_rows=$(hdfs dfs -cat "/user/hive/warehouse/$t/part-m-*" | wc -l)
mysql_rows=$(mysql -N -e "SELECT COUNT(*) FROM retail_db.$t")
echo "$t: HDFS=$hdfs_rows MySQL=$mysql_rows"
done

categories: HDFS=58 MySQL=58
customers: HDFS=12435 MySQL=12435
departments: HDFS=6 MySQL=6
order_items: HDFS=172198 MySQL=172198
order: HDFS=0 MySQL=
products: HDFS=1345 MySQL=1345


cat: `/user/hive/warehouse/order/part-m-*': No such file or directory
ERROR 1146 (42S02) at line 1: Table 'retail_db.order' doesn't exist


In [31]:
!hdfs dfs -du -h /user/hive/warehouse

1.0 K    1.0 K    /user/hive/warehouse/categories
931.2 K  931.2 K  /user/hive/warehouse/customers
60       60       /user/hive/warehouse/departments
5.2 M    5.2 M    /user/hive/warehouse/order_items
2.9 M    2.9 M    /user/hive/warehouse/orders
169.9 K  169.9 K  /user/hive/warehouse/products


In [32]:
!ls $CODEGEN_DIR/*.java

/content/cn7031-week3-sqoop/codegen/categories.java
/content/cn7031-week3-sqoop/codegen/customers.java
/content/cn7031-week3-sqoop/codegen/departments.java
/content/cn7031-week3-sqoop/codegen/order_items.java
/content/cn7031-week3-sqoop/codegen/orders.java
/content/cn7031-week3-sqoop/codegen/products.java


In [35]:
# Show the first 40 lines of the class generated for categories.
!head -n 40 $CODEGEN_DIR/categories.java

// ORM class for table 'categories'
// WARNING: This class is AUTO-GENERATED. Modify at your own risk.
//
// Debug information:
// Generated date: Tue Oct 20 16:28:38 UTC 2026
// For connector: cn7031.OrderedMySQLManager
import org.apache.hadoop.io.BytesWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.io.Writable;
import org.apache.hadoop.mapred.lib.db.DBWritable;
import com.cloudera.sqoop.lib.JdbcWritableBridge;
import com.cloudera.sqoop.lib.DelimiterSet;
import com.cloudera.sqoop.lib.FieldFormatter;
import com.cloudera.sqoop.lib.RecordParser;
import com.cloudera.sqoop.lib.BooleanParser;
import com.cloudera.sqoop.lib.BlobRef;
import com.cloudera.sqoop.lib.ClobRef;
import com.cloudera.sqoop.lib.LargeObjectLoader;
import com.cloudera.sqoop.lib.SqoopRecord;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.SQLException;
import java.io.DataInput;
import java.io.DataOutput;
import java.io.IOException;
import java.nio.ByteBuffer;
import java.n

In [36]:
# @title ▶ 4.4 Show an HDFS folder as the NameNode's "Browse Directory" page
# The HDFS folder to show. Type another path in the box on the right and run the cell again.
path = '/user/hive/warehouse'  # @param {type:"string"}

import html
import subprocess
from IPython.display import HTML, display

# Format a size in bytes the way the NameNode page does (B, KB, MB, GB).
def human(size):
    size = float(size)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if size < 1024 or unit == 'TB':
            return f'{size:.0f} {unit}' if unit == 'B' else f'{size:.2f} {unit}'
        size /= 1024

# 1. hdfs dfs -ls gives permission, replication, owner, group, size, date, time and path for each entry.
listing = subprocess.run(['hdfs', 'dfs', '-ls', path], capture_output=True, text=True)
rows = [line.split(None, 7) for line in listing.stdout.splitlines()
        if not line.startswith('Found ') and line.strip()]

if listing.returncode:
    print(listing.stderr.strip().splitlines()[-1])
elif not rows:
    print(f'{path} is empty.')
else:
    # 2. hdfs dfs -stat %o gives the block size of each entry (0 for a directory).
    blocks = subprocess.run(['hdfs', 'dfs', '-stat', '%o', *[r[7] for r in rows]],
                            capture_output=True, text=True).stdout.split()
    # Build one table row per entry, in the same columns as the web page.
    body = ''
    for (perm, repl, owner, group, size, day, clock, full), block in zip(rows, blocks):
        name = full.rstrip('/').rsplit('/', 1)[-1]
        body += (f'<tr><td>{perm}</td><td>{owner}</td><td>{group}</td><td>{human(size)}</td>'
                 f'<td>{day} {clock}</td><td>{0 if repl == "-" else repl}</td>'
                 f'<td>{human(block)}</td><td><b>{html.escape(name)}</b></td></tr>')
    head = ''.join(f'<th>{c}</th>' for c in
                   ('Permission', 'Owner', 'Group', 'Size', 'Last Modified', 'Replication', 'Block Size', 'Name'))
    display(HTML(f'''
<style>.hdfs-ui th, .hdfs-ui td {{padding:4px 10px; border-bottom:1px solid #8884}}</style>
<div class="hdfs-ui" style="font-family:sans-serif;max-width:900px">
  <h3 style="margin:0 0 6px">Browse Directory</h3>
  <div style="border:1px solid #999;padding:4px 8px;margin-bottom:8px;font-family:monospace">{html.escape(path)}</div>
  <table style="border-collapse:collapse;width:100%">
    <thead style="text-align:left;border-bottom:2px solid #999">{head}</thead>
    <tbody>{body}</tbody>
  </table>
  <p style="font-size:12px;opacity:.7">Showing {len(rows)} entries · built from hdfs dfs -ls and hdfs dfs -stat %o</p>
</div>'''))

drwxr-xr-x,root,supergroup,0 B,2026-10-20 16:28,0,0 B,categories
drwxr-xr-x,root,supergroup,0 B,2026-10-20 16:28,0,0 B,customers
drwxr-xr-x,root,supergroup,0 B,2026-10-20 16:28,0,0 B,departments
drwxr-xr-x,root,supergroup,0 B,2026-10-20 16:29,0,0 B,order_items
drwxr-xr-x,root,supergroup,0 B,2026-10-20 16:29,0,0 B,orders
drwxr-xr-x,root,supergroup,0 B,2026-10-20 16:29,0,0 B,products


In [53]:
# @title ▶ 5.1 Read the tables from HDFS, apply their schemas, and turn on %%sql cells
import csv
import io
import subprocess
import sys

import pandas as pd
from IPython.core.magic import register_cell_magic

# DuckDB is preinstalled in Colab; install it only if it is missing.
try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'], check=True)
    import duckdb

# The schema of each table: column names in the order Sqoop wrote them, and their types.
SCHEMAS = {
    'departments': {'department_id': 'int', 'department_name': 'text'},
    'categories':  {'category_id': 'int', 'category_department_id': 'int', 'category_name': 'text'},
    'products':    {'product_id': 'int', 'product_category_id': 'int', 'product_name': 'text',
                    'product_description': 'text', 'product_price': 'float', 'product_image': 'text'},
    'customers':   {'customer_id': 'int', 'customer_fname': 'text', 'customer_lname': 'text',
                    'customer_email': 'text', 'customer_password': 'text', 'customer_street': 'text',
                    'customer_city': 'text', 'customer_state': 'text', 'customer_zipcode': 'text'},
    'orders':      {'order_id': 'int', 'order_date': 'datetime', 'order_customer_id': 'int',
                    'order_status': 'text'},
    'order_items': {'order_item_id': 'int', 'order_item_order_id': 'int', 'order_item_product_id': 'int',
                    'order_item_quantity': 'int', 'order_item_subtotal': 'float',
                    'order_item_product_price': 'float'},
}

# Read all part files of one table from HDFS and apply its schema. Set delimiter default to '\t' matching our safe Sqoop setting.
def read_hdfs_table(table, directory='/user/hive/warehouse', delimiter='\t'):
    columns = SCHEMAS[table]
    raw = subprocess.run(['hdfs', 'dfs', '-cat', f'{directory}/{table}/part-m-*'],
                         capture_output=True, encoding='utf-8', errors='replace', check=True).stdout
    # Keep only real data lines: Sqoop ends every row with a newline, so ignore blank lines.
    lines = [line for line in raw.split('\n') if line.strip()]
    # 1. Split each line into columns and keep every value as text, exactly as stored in HDFS.
    frame = pd.read_csv(io.StringIO('\n'.join(lines)), sep=delimiter, header=None, names=list(columns),
                        dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False,
                        lineterminator='\n', on_bad_lines='skip')
    # 2. Convert each column to its type; a value that does not fit becomes missing instead of crashing.
    misfits = []
    for name, kind in columns.items():
        if kind == 'int':
            frame[name] = pd.to_numeric(frame[name], errors='coerce').astype('Int64')
        elif kind == 'float':
            frame[name] = pd.to_numeric(frame[name], errors='coerce')
        elif kind == 'datetime':   # Sqoop writes DATETIME values as 2013-07-25 00:00:00.0
            frame[name] = pd.to_datetime(frame[name], format='%Y-%m-%d %H:%M:%S.%f', errors='coerce')
        else:
            frame[name] = frame[name].astype('string')
        if kind != 'text' and frame[name].isna().any():
            misfits.append(f'{name} ({int(frame[name].isna().sum())})')
    # 3. Warn when values do not fit their type: the files are probably not in the expected column order.
    if misfits:
        print(f'  ! {table}: values that do not fit their type in ' + ', '.join(misfits) + '.')
        print(f'    First line: {lines[0][:120]}  <- check the column order and delimiter.')
    # 4. Every row must have its primary key (the first column); drop any line that does not.
    key = next(iter(columns))
    bad = frame[key].isna()
    skipped = len(lines) - len(frame) + int(bad.sum())
    if skipped:
        print(f'  ! {table}: {skipped} line(s) could not be read and were left out.')
    return frame[~bad].reset_index(drop=True)

# Load every table and register it with DuckDB under its own name.
con = duckdb.connect()
for table in SCHEMAS:
    frame = read_hdfs_table(table)
    con.register(table, frame)
    print(f'{table:<12} {len(frame):>8,} rows · {len(frame.columns)} columns')

# Notebook shortcut: a %%sql cell runs its SQL in DuckDB over the HDFS tables and shows the result.
@register_cell_magic
def sql(line, cell):
    result = con.sql(cell)
    return None if result is None else result.df()

print('\n%%sql cells are ready ✓')

departments         6 rows · 2 columns
categories         58 rows · 3 columns
products        1,345 rows · 6 columns
customers      12,435 rows · 9 columns
orders         68,883 rows · 4 columns
order_items   172,198 rows · 6 columns

%%sql cells are ready ✓


In [38]:
%%sql
SHOW TABLES;

,name
0,categories
1,customers
2,departments
3,order_items
4,orders
5,products


In [39]:
%%sql
DESCRIBE customers;


,column_name,column_type,null,key,default,extra
0,customer_id,BIGINT,YES,None,None,None
1,customer_fname,VARCHAR,YES,None,None,None
2,customer_lname,VARCHAR,YES,None,None,None
3,customer_email,VARCHAR,YES,None,None,None
4,customer_password,VARCHAR,YES,None,None,None
5,customer_street,VARCHAR,YES,None,None,None
6,customer_city,VARCHAR,YES,None,None,None
7,customer_state,VARCHAR,YES,None,None,None
8,customer_zipcode,VARCHAR,YES,None,None,None


In [43]:
%%sql
SELECT * FROM customers ORDER BY customer_id LIMIT 20;

,customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode


In [44]:
%%sql
SELECT order_status, COUNT (*) AS orders FROM orders GROUP BY order_status ORDER BY orders DESC;

,order_status,orders


In [42]:
%%sql
SELECT c.category_name, CAST(SUM(oi.order_item_quantity) AS BIGINT) AS units_sold
FROM order_items AS oi
JOIN products AS p ON oi.order_item_product_id = p.product_id
JOIN categories AS c ON p.product_category_id = c.category_id
GROUP BY c.category_name
ORDER BY units_sold DESC
LIMIT 10;

,category_name,units_sold


In [46]:
%%sql
SELECT p.product_id, p.product_name, ROUND(SUM(oi.order_item_subtotal),2) AS revenue
FROM order_items AS oi
JOIN orders AS o ON oi.order_item_order_id = o.order_id
JOIN products AS p ON oi.order_item_product_id = p.product_id
WHERE o.order_status NOT IN ('CANCELED', 'SUSPECTED_FRAUD')
GROUP BY p.product_id, p.product_name
ORDER BY revenue DESC
LIMIT 10;

,product_id,product_name,revenue


In [55]:
%%mysql
USE retail_db;
DROP TABLE IF EXISTS customers_export;
CREATE TABLE customers_export AS
SELECT customer_id, customer_fname, customer_lname, customer_email, customer_password, customer_street, customer_city, customer_state, customer_zipcode
FROM customers
WHERE 1=0;
SHOW TABLES;
SELECT COUNT(*) AS rows_in_customers_export FROM customers_export;

--------------
DROP TABLE IF EXISTS customers_export
--------------

Query OK, 0 rows affected, 1 warning

--------------
CREATE TABLE customers_export AS
SELECT customer_id, customer_fname, customer_lname, customer_email, customer_password, customer_street, customer_city, customer_state, customer_zipcode
FROM customers
WHERE 1=0
--------------

Query OK, 0 rows affected
Records: 0  Duplicates: 0  Warnings: 0

--------------
SHOW TABLES
--------------

+---------------------+
| Tables_in_retail_db |
+---------------------+
| categories          |
| customers           |
| customers_export    |
| departments         |
| order_items         |
| orders              |
| products            |
+---------------------+
7 rows in set

--------------
SELECT COUNT(*) AS rows_in_customers_export FROM customers_export
--------------

+--------------------------+
| rows_in_customers_export |
+--------------------------+
|                        0 |
+--------------------------+
1 row in set

Bye


In [58]:
%%bash
sqoop import \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--table customers \
--target-dir /retail_db/customers \
--delete-target-dir

2026-10-20 17:32:58,353 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 17:32:58,449 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 17:32:58,616 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
2026-10-20 17:32:58,617 INFO tool.CodeGenTool: Beginning code generation
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
2026-10-20 17:32:59,495 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `customers` AS t LIMIT 1
2026-10-20 17:32:59,572 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `customers` AS t LIMIT 1
2026-10-20 17:32:59,584 INFO orm.CompilationManager: HADOOP_MAPRED_HOME is /content/cn7031-week3-sqoop/hadoop-3.5.0
Note: /content/cn7031-week3-sqoop/codegen/customers.jav

In [59]:
!hdfs dfs -ls /retail_db/customers

Found 2 items
-rw-r--r--   1 root supergroup          0 2026-10-20 17:33 /retail_db/customers/_SUCCESS
-rw-r--r--   1 root supergroup     953525 2026-10-20 17:33 /retail_db/customers/part-m-00000


In [61]:
!hdfs dfs -cat /retail_db/customers/part-m-00000 |head -n 3

1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521
2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126
3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,00725
cat: Unable to write to output stream.


In [62]:
%%bash
mysql -e "TRUNCATE TABLE retail_db.customers_export;"
sqoop export \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--table customers_export \
--export-dir /retail_db/customers \

2026-10-20 17:34:41,563 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 17:34:41,626 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 17:34:41,777 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
2026-10-20 17:34:41,783 INFO tool.CodeGenTool: Beginning code generation
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
2026-10-20 17:34:42,539 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `customers_export` AS t LIMIT 1
2026-10-20 17:34:42,659 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `customers_export` AS t LIMIT 1
2026-10-20 17:34:42,674 INFO orm.CompilationManager: HADOOP_MAPRED_HOME is /content/cn7031-week3-sqoop/hadoop-3.5.0
Note: /content/cn7031-week3-sqoop/codegen

In [64]:
%%mysql
USE retail_db;
SELECT COUNT(*) AS exported_rows FROM customers_export;
SELECT * FROM customers_export ORDER BY customer_id DESC LIMIT 5;
SELECT COUNT(*) AS rows_missing_or_changed FROM customers AS c
LEFT JOIN customers_export AS e
USING (customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode)
WHERE e.customer_id IS NULL;

--------------
SELECT COUNT(*) AS exported_rows FROM customers_export
--------------

+---------------+
| exported_rows |
+---------------+
|         12435 |
+---------------+
1 row in set

--------------
SELECT * FROM customers_export ORDER BY customer_id DESC LIMIT 5
--------------

+-------------+----------------+----------------+----------------+-------------------+----------------------------+---------------+----------------+------------------+
| customer_id | customer_fname | customer_lname | customer_email | customer_password | customer_street            | customer_city | customer_state | customer_zipcode |
+-------------+----------------+----------------+----------------+-------------------+----------------------------+---------------+----------------+------------------+
|       12435 | Laura          | Horton         | XXXXXXXXX      | XXXXXXXXX         | 5736 Honey Downs           | Summerville   | SC             | 29483            |
|       12434 | Mary           | Mills    

In [65]:
%%bash
sqoop import \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--table orders \
--columns "order_id,order_date,order_status" \
--where "order_status = 'CLOSED' " \
-m 1 \
--target-dir /retail_db/orders_closed \
--delete-target-dir

2026-10-20 17:41:58,237 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 17:41:58,322 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 17:41:58,535 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
2026-10-20 17:41:58,535 INFO tool.CodeGenTool: Beginning code generation
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
2026-10-20 17:41:59,519 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `orders` AS t LIMIT 1
2026-10-20 17:41:59,611 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `orders` AS t LIMIT 1
2026-10-20 17:41:59,624 INFO orm.CompilationManager: HADOOP_MAPRED_HOME is /content/cn7031-week3-sqoop/hadoop-3.5.0
Note: /content/cn7031-week3-sqoop/codegen/orders.java uses or

In [66]:
!hdfs dfs -ls /retail_db/orders_closed

Found 2 items
-rw-r--r--   1 root supergroup          0 2026-10-20 17:42 /retail_db/orders_closed/_SUCCESS
-rw-r--r--   1 root supergroup     263220 2026-10-20 17:42 /retail_db/orders_closed/part-m-00000


In [68]:
!hdfs dfs -cat /retail_db/orders_closed/part-m-00000 |head -n 3

1,2013-07-25 00:00:00.0,CLOSED
4,2013-07-25 00:00:00.0,CLOSED
12,2013-07-25 00:00:00.0,CLOSED
cat: Unable to write to output stream.


In [71]:
%%bash
sqoop import \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--query 'SELECT o.order_id,o.order_status,c.customer_state
FROM orders AS o
JOIN customers AS c ON o.order_customer_id=c.customer_id WHERE $CONDITIONS' \
--split-by order_id \
-m 2 \
--target-dir /retail_db/orders_with_state \
--delete-target-dir

2026-10-20 17:48:48,382 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 17:48:48,456 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 17:48:48,602 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
2026-10-20 17:48:48,602 INFO tool.CodeGenTool: Beginning code generation
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
2026-10-20 17:48:49,406 INFO manager.SqlManager: Executing SQL statement: SELECT o.order_id,o.order_status,c.customer_state
FROM orders AS o
JOIN customers AS c ON o.order_customer_id=c.customer_id WHERE  (1 = 0) 
2026-10-20 17:48:49,440 INFO manager.SqlManager: Executing SQL statement: SELECT o.order_id,o.order_status,c.customer_state
FROM orders AS o
JOIN customers AS c ON o.order_customer_id=c.c

In [72]:
!hdfs dfs -cat /retail_db/orders_with_state/part-m-00000 | head -n 3

1,CLOSED,NC
2,PENDING_PAYMENT,IL
3,COMPLETE,CA
cat: Unable to write to output stream.


In [73]:
%%bash
sqoop import \
--connect "$RETAIL_JDBC" \
--username root \
--password cloudera \
--table order_items \
--fields-terminated-by '|' \
--compress \
--target-dir /retail_db/order_items_gz \
--delete-target-dir

2026-10-20 17:50:23,159 INFO sqoop.Sqoop: Running Sqoop version: 1.4.7
2026-10-20 17:50:23,265 WARN tool.BaseSqoopTool: Setting your password on the command-line is insecure. Consider using -P instead.
2026-10-20 17:50:23,506 INFO manager.MySQLManager: Preparing to use a MySQL streaming resultset.
2026-10-20 17:50:23,506 INFO tool.CodeGenTool: Beginning code generation
Loading class `com.mysql.jdbc.Driver'. This is deprecated. The new driver class is `com.mysql.cj.jdbc.Driver'. The driver is automatically registered via the SPI and manual loading of the driver class is generally unnecessary.
2026-10-20 17:50:24,627 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `order_items` AS t LIMIT 1
2026-10-20 17:50:24,720 INFO manager.SqlManager: Executing SQL statement: SELECT t.* FROM `order_items` AS t LIMIT 1
2026-10-20 17:50:24,739 INFO orm.CompilationManager: HADOOP_MAPRED_HOME is /content/cn7031-week3-sqoop/hadoop-3.5.0
Note: /content/cn7031-week3-sqoop/codegen/order_ite

In [75]:
!hdfs dfs -du -s -h /user/hive/warehouse/order_items /retail_db/order_items_gz

5.2 M     5.2 M     /user/hive/warehouse/order_items
1006.3 K  1006.3 K  /retail_db/order_items_gz


In [76]:
!hdfs dfs -text /retail_db/order_items_gz/part-m-00000.gz | head -n 3

1|1|957|1|299.98|299.98
2|2|1073|1|199.99|199.99
3|2|502|5|250.0|50.0
text: Unable to write to output stream.
